In [ ]:
pip install praw pandas regex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 11.9 MB/s eta 0:00:00


In [ ]:
import praw
import pandas as pd
import re
import os
from dotenv import load_dotenv

# 1. Wczytujemy plik .env
load_dotenv()

# 2. Inicjujemy połączenie z Reddit API używając bezpiecznych zmiennych
reddit = praw.Reddit(
    client_id=os.getenv("REDDIT_CLIENT_ID"),
    client_secret=os.getenv("REDDIT_CLIENT_SECRET"),
    user_agent="RAG_Scraper_Script_v1.0" # User agent nie musi być w .env, to tylko nazwa bota
)

# Test, czy działa (nie pokaże tajnych danych, tylko upewni się, że masz połączenie)
print("Połączono z Reddit jako:", reddit.config.user_agent)

THREAD_URL = "https://www.reddit.com/r/energy/comments/1e5luu1/why_does_the_rnc_seem_to_think_we_dont_produce/"

# ========== EMOJI ==========
emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+", flags=re.UNICODE
)

def clean_text(text):
    text = emoji_pattern.sub("", text)
    text = re.sub(r"http\S+", "", text)
    text = text.lower()
    return text

# ========== SŁOWNIKI PERSWAZJI ==========
SCIENCE = ["science", "study", "research", "data", "evidence", "scientists", "experts", "ipcc", "report"]
SOCIAL = ["we", "society", "people", "everyone", "most people", "community", "future generations"]
FEAR = ["fear", "threat", "danger", "cost", "expensive", "destroy", "ruin", "lose", "freedom", "collapse", "crisis"]

def count_terms(text, wordlist):
    return sum(text.count(w) for w in wordlist)

# ========== SUBMISSION ==========
submission = reddit.submission(url=THREAD_URL)
submission.comments.replace_more(limit=0)

data = []

for comment in submission.comments:
    if not isinstance(comment, praw.models.Comment):
        continue

    raw = comment.body
    text = clean_text(raw)

    science_score = count_terms(text, SCIENCE)
    social_score = count_terms(text, SOCIAL)
    fear_score = count_terms(text, FEAR)

    if max(science_score, social_score, fear_score) == science_score:
        persuasion = "science"
    elif max(science_score, social_score, fear_score) == social_score:
        persuasion = "social_norm"
    else:
        persuasion = "fear"

    data.append({
        "comment_id": comment.id,
        "username": comment.author.name if comment.author else "DELETED",
        "comment": raw,
        "score": comment.score,
        "reply_count": len(comment.replies),   # 🔴 H2
        "created_utc": comment.created_utc,
        "science_terms": science_score,
        "social_norm_terms": social_score,
        "fear_terms": fear_score,
        "persuasion_type": persuasion
    })

# ========== SAVE ==========
title = re.sub(r'[\\/*?:"<>|]', "", submission.title)
df = pd.DataFrame(data)
df.to_csv(f"{title}.csv", index=False)

print("Zapisano:", len(df), "komentarzy")

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Zapisano: 335 komentarzy
